Importing modules

In [18]:
import numpy as np
import matplotlib.pyplot as plt

Defining parameters

In [19]:
n_users = 35        # secondary users
n_primary = 8       # primary channels
n_channels = 40     # available spectrum bands

noise = 0.1         # noise for SNR

PS_penalty = 75     # penalty for primary-secondary user channel collision, arbitrary, put it after some raw testing, actual penalties must be figured out
SS_penalty = 25     # penalty for secondary-secondary user channel collision, arbitrary, put it after some raw testing, actual penalties must be figured out

N=200               # particles
S=n_channels-1      # search space [0, n_channels-1]
D=n_users           # dimensions = number of users
n_iter = 600        # number of updates of positions
s = 5               # seed inside both the functions

a=0.9               # parameters for PSO, according to my tuning
b=1.496
b_hat=1.496
c=0.5

beta_start = 0.25
beta_end = 0.65     # parameters for QPSO, according to my tuning, yes beta is increasing, that is what my results were showing


Getting SINR from random

In [20]:
# Simulate channel gains randomly (in reality this comes from pathloss model)
# SNR[i][j] = signal quality if user i uses channel j

channel_gain = np.random.uniform(0.1, 1.0, (n_users, n_channels))     # to get SNR which gets throughput
SNR = channel_gain / noise                                            # simplified, no inter-user interference yet
SNR


array([[7.99234904, 1.28627283, 1.45364837, ..., 1.95233249, 3.8650375 ,
        1.56264915],
       [7.00495731, 3.02186618, 8.73762909, ..., 8.32842975, 8.86105016,
        6.45937752],
       [2.11541087, 1.6274107 , 7.11502768, ..., 6.54970948, 2.65867116,
        2.34187855],
       ...,
       [1.05487418, 9.5847027 , 6.74282717, ..., 6.892887  , 7.33257377,
        8.74320604],
       [1.6454056 , 1.443318  , 6.36850298, ..., 8.34186066, 2.94624977,
        6.50329584],
       [4.84338462, 6.22134423, 4.9592017 , ..., 1.7177992 , 1.98879266,
        7.61926813]], shape=(35, 40))

Setting up Primary Users

In [21]:
# Primary user occupancy: PU[j] = 1 means channel j is occupied by primary user

PU_occupied = np.zeros(n_channels, dtype=int)
PU_occupied[np.random.choice(n_channels, n_primary, replace=False)] = 1
PU_occupied
 

array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1])

Fitness function which gets the throughput using SINR and adds penalty

In [22]:
def fitness(x, PS_penalty=PS_penalty, SS_penalty=SS_penalty):
    channels = np.clip(np.round(x).astype(int), 0, n_channels - 1)
    pu_mask = PU_occupied[channels]                         # 1 where PU is present
    sinr_vals = SNR[np.arange(n_users), channels]           # SINR for each user's channel
    throughput = np.sum(np.log2(1 + sinr_vals) * (1 - pu_mask))
    ps_penalty = np.sum(pu_mask) * PS_penalty
    
    counts = np.bincount(channels, minlength=n_channels)
    ss_pen = np.sum(np.maximum(counts - 1, 0)) * SS_penalty
    
    return -throughput + ps_penalty + ss_pen

PSO algorithm

In [23]:
def dpso(f, D, N, S, n_iter, a, b, b_hat, c, see):
    np.random.seed(see)
    x = np.random.uniform(0, S, size=(N, D))                 # setting up the initial positions of the N number of particles
    print(x)
    v = np.random.normal(size=(N, D))                        # setting up the initial velocities
    p = x.copy()                                             # best particle position
    fp = np.array([f(p[i]) for i in range(N)])               # throughput of all particles
    p_hat = x[np.argmin(fp)].copy()                          # global best position
    fp_hat = f(p_hat)                                        # throughput of global best position
    fp_hat_his = []

    for i in range (n_iter):
        fp_hat_his.append(float(fp_hat))
        #if i%(n_iter//10)== 0:                               # to show progress
            #print(f"progress {(i/n_iter)*100:.0f}%")
            #pass
            

        r,r_hat = np.random.uniform(0, 1, (2,N, D))          # setting up random parameters

        v = a*v + b*r*(p-x) + b_hat*r_hat*(p_hat-x)          # updating velocities as vector sum of the directions of initial velocity, local minima, local maxima
        x = x + c*v                                          # updating position according to velocities
        x = np.clip(x, 0, S)                                 # to limit the range within the subspace

        for n in range(N):                                   # calculation for each particle

            xn = x[n]                                        # getting position of that particle           
            fxn = f(xn)                                      # current throughput of that particle
            fpn = fp[n]                                      # best throughput of that particle

            if fxn < fpn:                                    # if current throughput of that particle is better the previous ones, update
                p[n] = xn.copy()
                fp[n] = fxn

                if fxn < fp_hat:                             # if the current througput is the global best throughput, update
                    p_hat = xn.copy()
                    fp_hat = fxn
    
    # print("progress 100%")
    return p_hat,fp_hat_his                                  # "coordinates", ie channel allocation of global best throughput along with global best history


Calling PSO 

In [24]:
result, history = dpso(fitness, D, N, S, n_iter, a, b, b_hat, c,s)
C_best_assignment = np.clip(np.round(result).astype(int), 0, n_channels-1)
C_throughput = 0
for user in range(n_users):
    ch = C_best_assignment[user]
    if not PU_occupied[ch]:
        C_throughput += np.log2(1 + SNR[user, ch])


[[ 8.65773367 33.95855994  8.06204706 ... 27.29390095 30.40106884
   0.89439061]
 [22.52885147  0.06404474 20.10343186 ...  1.38485224 21.30797559
  31.04956611]
 [ 1.99456932  7.3580417  14.25363295 ...  2.10452198  7.42863316
  17.64433499]
 ...
 [22.12073985 19.68398062 13.1837616  ...  5.40339527  3.07640493
  20.34202879]
 [ 6.47476509 14.59507338 20.47603992 ...  9.6490001  33.96522967
  38.04740693]
 [37.62011065 21.0267445  32.90144635 ... 14.08383168 35.09976618
   4.38271098]]


QPSO algorithm

In [25]:
def dqpso(f, D, N, S, n_iter, beta_start, beta_end,see):
    np.random.seed(see)
    x = np.random.uniform(0, S, size=(N, D))                            # setting up the initial positions of the N number of particles
    print(x)
    p = x.copy()                                                        # best particle position
    fp = np.array([f(p[i]) for i in range(N)])                          # throughput of all particles
    p_hat = x[np.argmin(fp)].copy()                                     # global best position
    fp_hat = f(p_hat)                                                   # throughput of global best position
    fp_hat_his = []

    for i in range(n_iter):
        fp_hat_his.append(float(fp_hat))
        #if i%(n_iter//10)== 0:                                          # to show progress
            #print(f"progress {(i/n_iter)*100:.0f}%")
        
        
        beta = beta_start - (beta_start - beta_end) * i / n_iter        # Beta increases linearly from beta_start to beta_end

        mbest = np.mean(p, axis=0)                                      # Mean best position ie average of all personal bests

        phi = np.random.uniform(0,1, (N,D))                             
        p_local = phi * p + (1 - phi) * p_hat                           # local attractor for each particle (works like velocity or inertia)

        u = np.random.uniform(1e-12, 1, (N,D))                           
        sign = 2 * np.random.randint(0, 2, size=(N,D)) - 1
        x = p_local + sign * beta * np.abs(mbest - x) * np.log(1/u)     # calculates the next value of x
        x = np.clip(x, 0, S)                                            # to limit the range within the subspace


        for n in range(N):                                              # calculation for each particle

            xn = x[n]                                                   # getting position of that particle           
            fxn = f(xn)                                                 # current throughput of that particle
            fpn = fp[n]                                                 # best throughput of that particle

            if fxn < fpn:                                               # if current throughput of that particle is better the previous ones, update
                p[n] = xn.copy()
                fp[n] = fxn

                if fxn < fp_hat:                                        # if the current througput is the global best throughput, update
                    p_hat = xn.copy()
                    fp_hat = fxn
    
    #print("progress 100%")
    return p_hat, fp_hat_his                                            # "coordinates", ie channel allocation of global best throughput along with global best history

            

    

Calling QPSO

In [26]:

result_qpso,qhistory = dqpso(fitness, D, N, S, n_iter, beta_start, beta_end,s)

Q_best_assignment = np.clip(np.round(result_qpso).astype(int), 0, n_channels-1)

Q_throughput = 0
for user in range(n_users):
    ch = Q_best_assignment[user]
    if not PU_occupied[ch]:
        Q_throughput += np.log2(1 + SNR[user, ch])




[[ 8.65773367 33.95855994  8.06204706 ... 27.29390095 30.40106884
   0.89439061]
 [22.52885147  0.06404474 20.10343186 ...  1.38485224 21.30797559
  31.04956611]
 [ 1.99456932  7.3580417  14.25363295 ...  2.10452198  7.42863316
  17.64433499]
 ...
 [22.12073985 19.68398062 13.1837616  ...  5.40339527  3.07640493
  20.34202879]
 [ 6.47476509 14.59507338 20.47603992 ...  9.6490001  33.96522967
  38.04740693]
 [37.62011065 21.0267445  32.90144635 ... 14.08383168 35.09976618
   4.38271098]]


In [27]:
print("Classical Particle Swarm Optimization\n")
print("Channel assignment:", C_best_assignment)
print("Throughput:", C_throughput)

print("\n\nQuantum Particle Swarm Optimization \n")
print("Channel assignment:", Q_best_assignment)
print("Throughput:", Q_throughput)


Classical Particle Swarm Optimization

Channel assignment: [18  1 38 34 13  2 14  3 12 32 22 23 28 17 29  8 21 11 35 16 11  4  7 26
 19 22  0 33 17 25  9  5 36 21 20]
Throughput: 96.79799294149831


Quantum Particle Swarm Optimization 

Channel assignment: [13  5 26 23 29 30 35  4 19 28 12 17 13 11 21 17 34 33 36  3 17  9 32 22
 14 25  8 20 18  2 16  1 25  7 16]
Throughput: 99.31339588302293
